In [0]:
# --- 3. Feature Selection & Multicollinearity Check (VIF) ---
print("\n--- Step 3: Feature Selection via VIF and Correlation Thresholds ---")

def calculate_vif(dataframe, features):
    """Calculates the Variance Inflation Factor for features."""
    vif_data = pd.DataFrame()
    vif_data["feature"] = features
    # .values is important here for the function to work correctly
    vif_data["VIF"] = [variance_inflation_factor(dataframe[features].values, i) 
                       for i in range(len(features))]
    return vif_data.sort_values(by="VIF", ascending=False)

# Check VIF for initial features
vif_results = calculate_vif(df, INITIAL_FEATURES)
print("Initial VIF Results (before selection):")
print(vif_results)

# A VIF > 5 indicates high multicollinearity. 'Radio_Spend' is highly correlated with 'Advertising_Spend'.
# We remove 'Radio_Spend' to address this issue.
features_to_model = ['Advertising_Spend', 'Experience_Years', 'Market_Share']

print(f"\nRemoved 'Radio_Spend' due to high VIF/redundancy. Final features: {features_to_model}\n")

# Verify VIFs after removal
final_vif_results = calculate_vif(df, features_to_model)
print("Final VIF Results (after selection):")
print(final_vif_results)

# --- 4. Regression Analysis (Model Building with statsmodels) ---
print("\n--- Step 4: Regression Model Building ---")

X = df[features_to_model]
y = df[TARGET_VAR]

# Add a constant (intercept) term
X = sm.add_constant(X)

# Fit the OLS model
model = sm.OLS(y, X).fit()

# Print the summary (R-squared, P-values, Coefficients)
print(model.summary())

# --- 5. Residual Analysis & Assumption Checking (Graphics Enhanced) ---
print("\n--- Step 5: Residual Analysis & Assumption Checking (Graphics Enhanced) ---")

residuals = model.resid
predicted_values = model.fittedvalues

# A. Linearity and Homoscedasticity Check (Residuals vs. Fitted Plot)
plt.figure(figsize=(10, 6))
sns.scatterplot(x=predicted_values, y=residuals)
plt.axhline(y=0, color='red', linestyle='--', lw=2)
plt.xlabel('Fitted values (Predicted Y)')
plt.ylabel('Residuals (Errors)')
plt.title('Residuals vs. Fitted Values Plot (Check for Patterns/Variance)')
plt.show()

# Formal tests for confirmation:
rainbow_test_stat, rainbow_p_value = linear_rainbow(model)
print(f"Rainbow Test for Linearity: P-value = {rainbow_p_value:.4f}")
bp_test_stat, bp_p_value, _, _ = het_breuschpagan(residuals, model.model.exog)
print(f"Breusch-Pagan Test for Homoscedasticity: P-value = {bp_p_value:.4f}\n")

# B. Normality of Residuals Check (Q-Q Plot and Histogram)
# Histogram
plt.figure(figsize=(10, 6))
sns.histplot(residuals, kde=True, bins=20)
plt.title('Histogram of Residuals (Check for Normality)')
plt.xlabel('Residual Value')
plt.show()

# Q-Q Plot is a powerful visual for normality
sm.qqplot(residuals, line='s')
plt.title('Normal Q-Q Plot of Residuals')
plt.show()

print("\n--- Bilateral Regression Analysis Complete ---")
